In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install datasets accelerate bert-score pyarrow

## **01. Read data - Config**

In [ ]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"
RUN_NAME = "Qwen3-0.6B"


ENABLE_THINKING = False

NUMBER_OF_EPOCHS = 10
LEARNING_RATE = 2e-5
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CHECKPOINTING = True
WARMUP_RATIO = 0.1
RANDOM_SEED = 42

MAX_SOURCE_LENGTH = 2048
MAX_TARGET_LENGTH = 1024

GENERATION_BATCH_SIZE = 32
NUM_BEAMS = 1

INCLUDE_DOCUMENT_NAME = False
CONTEXT_SEPARATOR = '\n\n'
TOKENIZE_KWARGS = dict(lowercase=True, strip_punctuation=True)

USE_BERTSCORE = True
BERTSCORE_MODEL = None

USE_CORPUS_AS_CONTEXT_SOURCE = True

HYBRID_RUN_NAME = 'hybrid__bge-m3__marginmse-cross_encoder'

RETRIEVAL_TOP_K = 1

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"
GENERATOR_DIR = BASE_DIR / "03_Generator"

HYBRID_OUTPUT_DIR = RETRIEVER_DIR / "04_Hybrid/output"
RETRIEVER_RANKING_PATH = HYBRID_OUTPUT_DIR / f'{HYBRID_RUN_NAME}__test_ranking.parquet'

OUTPUT_DIR = GENERATOR_DIR / "output_1"
CHECKPOINT_DIR = GENERATOR_DIR / "checkpoints_1" / RUN_NAME
MODEL_SAVE_DIR = GENERATOR_DIR / "models_1" / RUN_NAME

for directory in [OUTPUT_DIR, CHECKPOINT_DIR, MODEL_SAVE_DIR]:
  directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(GENERATOR_DIR))

import importlib
import generator_common

importlib.reload(generator_common)

gc = generator_common

gc.require_version(5)

torch.manual_seed(RANDOM_SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



generator_common v5 (need >= 5)
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [ ]:
qa_master = pd.read_parquet(DATA_DIR / 'corpus' / 'qa_master.parquet')
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')

context_by_doc = dict(zip(corpus['doc_id'], corpus['context'].astype(str)))

context_mismatch = sum(context_by_doc.get(doc_id) != context for doc_id, context in zip(qa_master['doc_id'], qa_master['context'].astype(str)))

print(f"Dòng qa_master có context khác corpus: {context_mismatch:,} / {len(qa_master):,}")

if context_mismatch:
  print("=> Bắt buộc dùng corpus làm nguồn context, và PHẢI fine-tune lại.")
else:
  print("=> Hai nguồn trùng nhau. Đổi sang corpus chỉ để code sạch, không cần train lại.")

data = {
    split_name: gc.build_generator_dataframe(qa_master, split_name, include_document_name=INCLUDE_DOCUMENT_NAME, corpus=corpus if USE_CORPUS_AS_CONTEXT_SOURCE else None)
    for split_name in ['train', 'val', 'test']
}

for split_name, frame in data.items():
  print(f"{split_name}: {len(frame):,} mẫu")

Dòng qa_master có context khác corpus: 3 / 9,609
=> Bắt buộc dùng corpus làm nguồn context, và PHẢI fine-tune lại.
train: 7,718 mẫu
val: 933 mẫu
test: 958 mẫu


In [ ]:
data['train']['target'].str.split().str.len().describe()

,target
count,7718.000000
mean,34.743068
std,39.919414
min,2.000000
25%,18.000000
50%,25.000000
75%,37.000000
max,729.000000


In [ ]:
for split_name, frame in data.items():
  empty_targets = frame['target'].str.strip().isin(['', 'nan', 'None', 'NaN']).sum()

  if empty_targets:
    print(f"Warning: {split_name}: {empty_targets} mẫu có target rỗng/'nan'.")

print("\nMẫu đầu tiên của train:")

row = data['train'].iloc[0]

print("\n- question:", row['source_question'])
print("\n- context :", row['source_context'][:200], "...")
print("\n- target  :", row['target'])

print("\nThống kê chiều dài question + context:")
for split_name, frame in data.items():
  word_counts = (frame['source_question'] + ' ' + frame['source_context']).str.split().str.len()

  print(f"{split_name}: mean {word_counts.mean():.0f} từ, p95 {word_counts.quantile(0.95):.0f}, max {word_counts.max()}")


Mẫu đầu tiên của train:

- question: Sinh viên dự bị không trở thành sinh viên chính thức bao nhiêu học kỳ sẽ bị loại khỏi CTTN?

- context : Điều  9.	Tuyển bổ sung và loại ra khỏi chương trình, xét chính thức và dự bị
Đối tượng tham gia CTTN là những sinh viên có năng lực xuất sắc, do đó, sau mỗi học kỳ BĐH quyết định việc loại sinh viên k ...

- target  : Sinh viên dự bị không trở thành sinh viên chính thức 02 học kỳ liên tiếp sẽ bị loại khỏi CTTN.

Thống kê chiều dài question + context:
train: mean 388 từ, p95 958, max 1300
val: mean 379 từ, p95 925, max 1269
test: mean 392 từ, p95 970, max 1276


## **02. Model & Tokenizer**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

if torch.cuda.is_available():
  model = model.cuda()

print("Model:", MODEL_NAME)
print(f"Tham số: {sum(p.numel() for p in model.parameters()):,}")
print("pad_token:", repr(tokenizer.pad_token))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model: Qwen/Qwen3-0.6B
Tham số: 596,049,920
pad_token: '<|endoftext|>'


## **03. Prompt**

In [ ]:
qa_master['embedding_text'].iloc[0]

'Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG\nĐiều  9.\tTuyển bổ sung và loại ra khỏi chương trình, xét chính thức và dự bị\nĐối tượng tham gia CTTN là những sinh viên có năng lực xuất sắc, do đó, sau mỗi học kỳ BĐH quyết định việc loại sinh viên khỏi lớp tài năng, tuyển bổ sung sinh viên từ chương trình chuẩn vào lớp tài năng, xét chuyển đổi sinh viên chính thức và dự bị.\nĐầu mỗi học kỳ, Khoa xét và đề nghị lên BĐH các danh sách sinh viên tuyển bổ sung, bị loại ra khỏi các lớp CTTN hoặc danh sách sinh viên chính thức và dự bị theo các tiêu chuẩn như sau:\n1.\tLoại khỏi chương trình\nTại thời điểm xem xét, sinh viên rơi vào một trong các trường hợp sau:\n-\tChưa tốt nghiệp khi đã quá thời gian thiết kế của khóa học và không có lý do đặc biệt.\n-\tKhông đăng ký học đầy đủ các môn học CTTN bắt buộc trong học kỳ.\n-\tĐTBTL nhỏ hơn 6,5 – tính tương ứng sau học kỳ 1 và sau học kỳ hè (kết quả học tập của học kỳ hè sẽ được tính chung vào kết quả học tập học kỳ 2 của năm học tương ứng).\n

In [ ]:
import re

PROMPT_CONFIG = {
    'system_prompt': (
        "Bạn hãy trả lời câu hỏi về quy định đào tạo đại học dựa trên điều luật "
        "được cung cấp. Yêu cầu: chỉ dùng thông tin trong điều luật, không thêm "
        "thông tin mới và trả lời ngắn gọn bằng một câu hoàn chỉnh."
    ),
    'user_template': "Câu hỏi: {question}\nĐiều luật: {context}",
    'enable_thinking': ENABLE_THINKING,
    'include_document_name': INCLUDE_DOCUMENT_NAME,
    'context_separator': CONTEXT_SEPARATOR,
    'max_source_length': MAX_SOURCE_LENGTH,
    'max_target_length': MAX_TARGET_LENGTH,
    'num_beams': NUM_BEAMS,
    'do_sample': False,
    'model_name': MODEL_NAME,
    'context_source': 'corpus' if USE_CORPUS_AS_CONTEXT_SOURCE else 'qa_master',
}

In [ ]:
def build_user_message(question: str, context: str) -> str:
  return PROMPT_CONFIG['user_template'].format(question=str(question).strip(), context=str(context).strip())


def render_prompt_from_parts(question: str, context: str) -> str:
  messages = [
      {'role': 'system', 'content': PROMPT_CONFIG['system_prompt']},
      {'role': 'user', 'content': build_user_message(question, context)},
  ]

  template_kwargs = dict(tokenize=False, add_generation_prompt=True)

  if PROMPT_CONFIG['enable_thinking'] is not None:
    template_kwargs['enable_thinking'] = PROMPT_CONFIG['enable_thinking']

  try:
    return tokenizer.apply_chat_template(messages, **template_kwargs)
  except Exception:
    merged = [{'role': 'user',
               'content': PROMPT_CONFIG['system_prompt'] + "\n\n"
                          + build_user_message(question, context)}]

    return tokenizer.apply_chat_template(merged, **template_kwargs)

def render_prompt(row) -> str:
  """Giữ chữ ký cũ để cell Dataset/Generation không phải sửa."""
  return render_prompt_from_parts(row['source_question'], row['source_context'])


In [ ]:
# Smoke test
sample_prompt = render_prompt(data['train'].iloc[0])

# Workaround: remove <think> block if tokenizer incorrectly adds it
import re
if PROMPT_CONFIG['enable_thinking'] is False:
  sample_prompt = re.sub(r'<think>.*?</think>', '', sample_prompt, flags=re.DOTALL)


if PROMPT_CONFIG['enable_thinking'] is False:
  assert '<think>' not in sample_prompt, (
      "Chat template vẫn chèn <think> — kiểm tra lại enable_thinking."
  )

assert data['train'].iloc[0]['source_question'][:20] in sample_prompt
assert data['train'].iloc[0]['source_context'][:30] in sample_prompt

all_prompt_lengths = [
    len(tokenizer.encode(render_prompt(row), add_special_tokens=False))
    for row in data['train'].to_dict('records')
]

all_target_lengths = [
    len(tokenizer.encode(target, add_special_tokens=False)) for target in data['train']['target']
]

print(f"prompt dài nhất: {max(all_prompt_lengths)} / MAX_SOURCE_LENGTH = {MAX_SOURCE_LENGTH}")
print(f"target dài nhất: {max(all_target_lengths)} / MAX_TARGET_LENGTH = {MAX_TARGET_LENGTH}")

assert sum(1 for length in all_prompt_lengths if length > MAX_SOURCE_LENGTH) == 0
assert sum(1 for length in all_target_lengths if length > MAX_TARGET_LENGTH) == 0

print("\nSmoke test đã pass.")

prompt dài nhất: 1858 / MAX_SOURCE_LENGTH = 2048
target dài nhất: 1019 / MAX_TARGET_LENGTH = 1024

Smoke test đã pass.


In [ ]:
import re

if ENABLE_THINKING is False:
  sample_prompt = re.sub(r'<think>.*?</think>', '', sample_prompt, flags=re.DOTALL)

  assert '<think>' not in sample_prompt, ("Chat template vẫn chèn <think> — kiểm tra lại enable_thinking. Nếu không tắt được, phải lọc khối <think>...</think> khỏi output trước khi tính chỉ số.")
  print("Không có <think> trong prompt.")

assert data['train'].iloc[0]['source_question'][:20] in sample_prompt
assert data['train'].iloc[0]['source_context'][:30] in sample_prompt

all_prompt_lengths = [
    len(tokenizer.encode(render_prompt(row), add_special_tokens=False))
    for row in data['train'].to_dict('records')
]

all_target_lengths = [
    len(tokenizer.encode(target, add_special_tokens=False)) for target in data['train']['target']
]

truncated_source = sum(1 for length in all_prompt_lengths if length > MAX_SOURCE_LENGTH)
truncated_target = sum(1 for length in all_target_lengths if length > MAX_TARGET_LENGTH)

print(f"prompt dài nhất: {max(all_prompt_lengths)} / MAX_SOURCE_LENGTH = {MAX_SOURCE_LENGTH}")
print(f"target dài nhất: {max(all_target_lengths)} / MAX_TARGET_LENGTH = {MAX_TARGET_LENGTH}")
print(f"bị cắt: {truncated_source} prompt, {truncated_target} target")

assert truncated_source == 0 and truncated_target == 0, (
    "Có mẫu bị cắt cụt. Tăng MAX_SOURCE_LENGTH / MAX_TARGET_LENGTH — "
    "padding là động nên việc tăng trần gần như không tốn thêm bộ nhớ."
)

print("\nSmoke test đã pass.")

Không có <think> trong prompt.
prompt dài nhất: 1858 / MAX_SOURCE_LENGTH = 2048
target dài nhất: 1019 / MAX_TARGET_LENGTH = 1024
bị cắt: 0 prompt, 0 target

Smoke test đã pass.


## **04. Dataset + mask loss**

In [ ]:
from torch.utils.data import Dataset


class GeneratorDataset(Dataset):

  def __init__(self, frame: pd.DataFrame):
    self.examples = []

    for row in frame.to_dict('records'):
      prompt = render_prompt(row)

      prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)[-MAX_SOURCE_LENGTH:]

      target_ids = tokenizer.encode(row['target'], add_special_tokens=False)[:MAX_TARGET_LENGTH]
      target_ids = target_ids + [tokenizer.eos_token_id]

      input_ids = prompt_ids + target_ids

      labels = [-100] * len(prompt_ids) + target_ids

      self.examples.append({'input_ids': input_ids, 'labels': labels})

  def __len__(self):
    return len(self.examples)

  def __getitem__(self, index):
    return self.examples[index]

In [ ]:
def collate(batch):
  max_length = max(len(item['input_ids']) for item in batch)

  input_ids, labels, attention_mask = [], [], []

  for item in batch:
    padding = max_length - len(item['input_ids'])

    input_ids.append(item['input_ids'] + [tokenizer.pad_token_id] * padding)
    labels.append(item['labels'] + [-100] * padding)
    attention_mask.append([1] * len(item['input_ids']) + [0] * padding)

  return {
      'input_ids': torch.tensor(input_ids),
      'labels': torch.tensor(labels),
      'attention_mask': torch.tensor(attention_mask),
  }


train_dataset = GeneratorDataset(data['train'])
validation_dataset = GeneratorDataset(data['val'])

print(f"train: {len(train_dataset):,} | val: {len(validation_dataset):,}")

train: 7,718 | val: 933


In [ ]:
example = train_dataset[0]

masked_count = sum(1 for label in example['labels'] if label == -100)
kept_count = len(example['labels']) - masked_count

print(f"Token bị mask (prompt): {masked_count} | token tính loss (target): {kept_count}")

decoded_target = tokenizer.decode([label for label in example['labels'] if label != -100], skip_special_tokens=True)

print("Phần tính loss giải mã ra:", repr(decoded_target))

assert kept_count > 0, "Không có token nào được tính loss."
assert data['train'].iloc[0]['target'][:20] in decoded_target, ("Phần tính loss không khớp target — mask bị lệch.")

print("\nMask loss đúng.")

Token bị mask (prompt): 1175 | token tính loss (target): 28
Phần tính loss giải mã ra: 'Sinh viên dự bị không trở thành sinh viên chính thức 02 học kỳ liên tiếp sẽ bị loại khỏi CTTN.'

Mask loss đúng.


## **05. Training**

In [ ]:
from transformers import Trainer, TrainingArguments
import math

num_update_steps_per_epoch = math.ceil(len(train_dataset) / (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))

num_training_steps = num_update_steps_per_epoch * NUMBER_OF_EPOCHS
warmup_steps = int(num_training_steps * WARMUP_RATIO)

training_arguments = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUMBER_OF_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps, # Changed from warmup_ratio
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={'use_reentrant': False} if GRADIENT_CHECKPOINTING else None,

    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=RANDOM_SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collate,
)

start_time = time.perf_counter()

trainer.train()

training_seconds = time.perf_counter() - start_time

print(f"\nHuấn luyện xong trong {training_seconds / 60:.1f} phút")

Epoch,Training Loss,Validation Loss
1,0.229778,0.241006
2,0.242023,0.214613
3,0.174117,0.216003
4,0.117963,0.229417
5,0.080850,0.257418
6,0.055796,0.285165
7,0.045169,0.313497
8,0.030842,0.335964


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



Huấn luyện xong trong 124.7 phút


In [ ]:
model.save_pretrained(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

print("Đã lưu:", MODEL_SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu: /content/drive/MyDrive/Legal_Sytem/03_Generator/models_1/Qwen3-0.6B


## **06. Generation**

In [ ]:
def clean_generation(text: str) -> str:
  """Lọc khối <think> nếu model vẫn sinh ra, và bỏ khoảng trắng thừa."""
  text = re.sub(r'<think>.*?</think>', ' ', text, flags=re.DOTALL)
  text = re.sub(r'\s+', ' ', text)

  return text.strip()

In [ ]:
@torch.no_grad()
def generate_answers(frame: pd.DataFrame, batch_size: int = GENERATION_BATCH_SIZE) -> list[str]:
  model.eval()

  original_padding_side = tokenizer.padding_side
  original_truncation_side = tokenizer.truncation_side

  tokenizer.padding_side = 'left'        
  tokenizer.truncation_side = 'left'     

  prompts = [render_prompt(row) for row in frame.to_dict('records')]

  outputs = []

  for start in range(0, len(prompts), batch_size):
    batch_prompts = prompts[start:start + batch_size]

    encoded = tokenizer(batch_prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH, add_special_tokens=False)

    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    generated = model.generate(**encoded, max_new_tokens=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

    new_tokens = generated[:, encoded['input_ids'].shape[1]:]

    outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

  tokenizer.padding_side = original_padding_side
  tokenizer.truncation_side = original_truncation_side

  return [clean_generation(text) for text in outputs]

In [ ]:
predictions_by_split = {}
per_example_by_split = {}
summary_by_split = {}

for split_name in ['val', 'test']:
  frame = data[split_name]

  start_time = time.perf_counter()

  hypotheses = generate_answers(frame)

  generation_seconds = time.perf_counter() - start_time

  references = frame['target'].tolist()

  per_example = gc.evaluate_generations(references, hypotheses, frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

  summary = gc.summarize_generations(references, hypotheses, per_example, **TOKENIZE_KWARGS)

  summary['latency_ms_per_example'] = round(generation_seconds / len(frame) * 1000, 2)

  predictions_by_split[split_name] = frame.assign(prediction=hypotheses)
  per_example_by_split[split_name] = per_example
  summary_by_split[split_name] = summary

  print(f"[{split_name}] " + "  ".join(f"{key}={value}" for key, value in summary.items()))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

[val] BLEU1=75.24  BLEU4=69.32  ROUGE-L=75.25  F1=79.54  BERTScore=91.1500015258789  latency_ms_per_example=404.09


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[test] BLEU1=72.77  BLEU4=65.72  ROUGE-L=73.83  F1=78.36  BERTScore=90.75  latency_ms_per_example=541.29


### **6.1. Compare to baseline**

In [ ]:
comparison_records = []

for split_name in ['val', 'test']:
  frame = data[split_name]
  references = frame['target'].tolist()

  for baseline_name, baseline_hypotheses in gc.trivial_baselines(frame).items():
    baseline_per_example = gc.evaluate_generations(references, baseline_hypotheses, frame['qa_id'].tolist(),use_bertscore=False, **TOKENIZE_KWARGS)

    baseline_summary = gc.summarize_generations(references, baseline_hypotheses, baseline_per_example, **TOKENIZE_KWARGS)

    comparison_records.append({'split': split_name, 'hệ_thống': baseline_name, **baseline_summary})

  model_summary = {key: value for key, value in summary_by_split[split_name].items() if key != 'latency_ms_per_example'}

  comparison_records.append({'split': split_name, 'hệ_thống': RUN_NAME, **model_summary})

comparison_table = pd.DataFrame(comparison_records)

display(comparison_table)

,split,hệ_thống,BLEU1,BLEU4,ROUGE-L,F1,BERTScore
0,val,copy_question,23.99,20.75,54.66,59.45,NaN
1,val,copy_context,10.05,8.58,24.07,23.25,NaN
2,val,Qwen3-0.6B,75.24,69.32,75.25,79.54,91.150002
3,test,copy_question,27.78,23.69,54.83,60.06,NaN
4,test,copy_context,8.90,7.41,22.12,21.33,NaN
5,test,Qwen3-0.6B,72.77,65.72,73.83,78.36,90.750000


In [ ]:
test_frame = data['test']
test_references = test_frame['target'].tolist()

strong_baseline_name = 'copy_question'

strong_baseline = gc.trivial_baselines(test_frame)[strong_baseline_name]

baseline_per_example = gc.evaluate_generations(test_references, strong_baseline, test_frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

bootstrap_records = []

metrics = ['rouge_l', 'token_f1', 'sentence_bleu1']

if USE_BERTSCORE:
  metrics.append('bertscore')

for metric in metrics:
  result = gc.paired_bootstrap(per_example_by_split['test'], baseline_per_example, metric=metric)

  bootstrap_records.append({
      'metric': metric,
      RUN_NAME: round(result['mean_a'] * 100, 2),
      strong_baseline_name: round(result['mean_b'] * 100, 2),
      'chênh_lệch': round(result['difference'] * 100, 2),
      'ci_95': f"[{result['ci_lower_95'] * 100:+.2f}, {result['ci_upper_95'] * 100:+.2f}]",
      'p_value': round(result['p_value'], 4),
      'significant': result['significant_at_0.05'],
  })

display(pd.DataFrame(bootstrap_records))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,metric,Qwen3-0.6B,copy_question,chênh_lệch,ci_95,p_value,significant
0,rouge_l,73.83,54.83,19.01,"[+17.24, +20.77]",0.0,True
1,token_f1,78.36,60.06,18.30,"[+16.66, +19.95]",0.0,True
2,sentence_bleu1,71.81,46.28,25.53,"[+23.44, +27.61]",0.0,True
3,bertscore,90.75,82.16,8.59,"[+8.01, +9.17]",0.0,True


### **6.1. End2end evaluation (vs Retrieved context)**

In [ ]:

RETRIEVER_RANKING_PATH = Path(BASE_DIR / '02_Retriever/03_Finetuning/output/Qwen3-Embedding-0.6B__marginmse-cross_encoder__test_ranking.parquet')

ranking = pd.read_parquet(RETRIEVER_RANKING_PATH)

assert {'query_id', 'doc_id', 'rank'} <= set(ranking.columns), (f"Đây không phải file ranking. Cột hiện có: {list(ranking.columns)}")

missing_queries = set(data['test']['query_id']) - set(ranking['query_id'])

assert not missing_queries, f"{len(missing_queries)} câu hỏi test không có trong ranking."

end_to_end_frame = gc.attach_retrieved_context(data['test'], ranking, corpus, top_k=RETRIEVAL_TOP_K, include_document_name=INCLUDE_DOCUMENT_NAME, context_separator=CONTEXT_SEPARATOR)

print(f"Retriever hit@{RETRIEVAL_TOP_K}: {end_to_end_frame['retrieved_hit'].mean() * 100:.2f}%")

end_to_end_hypotheses = generate_answers(end_to_end_frame)
end_to_end_references = end_to_end_frame['target'].tolist()

end_to_end_per_example = gc.evaluate_generations(end_to_end_references, end_to_end_hypotheses, end_to_end_frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

end_to_end_per_example['retrieved_hit'] = end_to_end_frame['retrieved_hit'].to_numpy()

records = []

for label, mask in [
    ('context chuẩn (oracle)', None),
    (f'end-to-end top-{RETRIEVAL_TOP_K}', slice(None)),
    ('  trong đó Retriever ĐÚNG', end_to_end_per_example['retrieved_hit']),
    ('  trong đó Retriever SAI', ~end_to_end_per_example['retrieved_hit']),
]:
  if mask is None:
    records.append({'nhóm': label, 'n': len(data['test']), **{key: value for key, value in summary_by_split['test'].items() if key != 'latency_ms_per_example'}})
    continue

  subset = end_to_end_per_example if isinstance(mask, slice) else end_to_end_per_example[mask]
  indices = subset.index.to_numpy()

  records.append({
      'nhóm': label, 'n': len(subset),
      **gc.summarize_generations(
          [end_to_end_references[i] for i in indices],
          [end_to_end_hypotheses[i] for i in indices],
          subset, **TOKENIZE_KWARGS),
  })

display(pd.DataFrame(records))

predictions_by_split['test_end_to_end'] = end_to_end_frame.assign(prediction=end_to_end_hypotheses)
per_example_by_split['test_end_to_end'] = end_to_end_per_example
summary_by_split['test_end_to_end'] = gc.summarize_generations(
    end_to_end_references, end_to_end_hypotheses, end_to_end_per_example, **TOKENIZE_KWARGS)

Retriever hit@1: 59.29%


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,nhóm,n,BLEU1,BLEU4,ROUGE-L,F1,BERTScore
0,context chuẩn (oracle),958,72.77,65.72,73.83,78.36,90.750000
1,end-to-end top-1,958,57.52,49.10,63.45,68.08,87.269997
2,trong đó Retriever ĐÚNG,568,68.52,61.80,73.95,78.39,90.760002
3,trong đó Retriever SAI,390,40.50,29.11,48.16,53.08,82.199997


In [ ]:
sample = predictions_by_split['test'].sample(5, random_state=RANDOM_SEED)

for row in sample.itertuples(index=False):
  print("Câu hỏi   :", row.source_question)
  print("Điều luật :", row.source_context[:250], "...")
  print("Model     :", row.prediction)
  print("Chuẩn     :", row.target)
  print("-" * 90)

Câu hỏi   : Tổ soạn thảo đề án mở ngành đào tạo lập dự trù kinh phí căn cứ theo Quy chế chi tiêu nội bộ và tình hình thực tế thực hiện đúng không?
Điều luật : Điều  6.        Xây dựng đề án mở ngành đào tạo
1.        Giao nhiệm vụ xây dựng và triển khai đề án mở ngành đào tạo
Hiệu trưởng ra quyết định giao nhiệm vụ xây dựng và triển khai đề án mở ngành đào tạo cho ĐVCM hoặc nhóm chuyên gia.  
2.        Thà ...
Model     : Tổ soạn thảo đề án mở ngành đào tạo lập dự trù kinh phí căn cứ theo Quy chế chi tiêu nội bộ và tình hình thực tế thực hiện.
Chuẩn     : Tổ soạn thảo đề án mở ngành đào tạo lập dự trù kinh phí căn cứ theo Quy chế chi tiêu nội bộ và tình hình thực tế thực hiện.
------------------------------------------------------------------------------------------
Câu hỏi   : Đại diện các Khoa mở ngành đào tạo CTTN có làm ủy viên thường trực BĐH được không?
Điều luật : Điều  4.	Cơ cấu tổ chức
BĐH cấp trường (sau đây gọi là BĐH) quản lý CTTN dưới sự chỉ đạo của Hiệu trưởng. BĐH do Hiệ

## **07. Save**

In [ ]:
with open(MODEL_SAVE_DIR / 'prompt_config.json', 'w', encoding='utf-8') as file:
  json.dump(PROMPT_CONFIG, file, ensure_ascii=False, indent=2)

print("Đã lưu prompt_config.json:", MODEL_SAVE_DIR / 'prompt_config.json')

run_config = {
    'run_name': RUN_NAME,
    'model_name': MODEL_NAME,
    'architecture': 'decoder-only',
    'model_save_dir': str(MODEL_SAVE_DIR),
    'prompt_config': PROMPT_CONFIG,
    'context_source': PROMPT_CONFIG['context_source'],
    'retriever_ranking_path': str(RETRIEVER_RANKING_PATH),
    'retrieval_top_k': RETRIEVAL_TOP_K,
    'number_of_epochs': NUMBER_OF_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'effective_batch_size': PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'gradient_checkpointing': GRADIENT_CHECKPOINTING,
    'training_method': 'full',
    'generation_batch_size': GENERATION_BATCH_SIZE,
    'tokenize_kwargs': TOKENIZE_KWARGS,
    'generator_common_version': gc.MODULE_VERSION,
    'training_minutes': round(training_seconds / 60, 2),
    'peak_vram_gb': (round(torch.cuda.max_memory_allocated() / 1e9, 2)
                     if torch.cuda.is_available() else None),
}

summary_dataframe = gc.save_run(OUTPUT_DIR, RUN_NAME, predictions_by_split, per_example_by_split, summary_by_split, run_config)

display(summary_dataframe)

Đã lưu prompt_config.json: /content/drive/MyDrive/Legal_Sytem/03_Generator/models_1/Qwen3-0.6B/prompt_config.json


,run_name,split,BLEU1,BLEU4,ROUGE-L,F1,BERTScore,latency_ms_per_example
0,Qwen3-0.6B,val,75.24,69.32,75.25,79.54,91.150002,404.09
1,Qwen3-0.6B,test,72.77,65.72,73.83,78.36,90.750000,541.29
2,Qwen3-0.6B,test_end_to_end,57.52,49.10,63.45,68.08,87.269997,NaN


In [ ]:
leaderboard = gc.collect_leaderboard(OUTPUT_DIR)

display(leaderboard[leaderboard['split'] == 'test'])

,run_name,split,BLEU1,BLEU4,ROUGE-L,BERTScore,F1
0,Qwen3-0.6B,test,72.77,65.72,73.83,90.75,78.36
